# 06 — Fast End-to-End Demo
**Knowledge Distillation Combined with Quantization for Compact Models**

Runs M1 → M5 in one Colab session (~25–35 min on T4 GPU).

| Model | Architecture | Precision | Strategy |
|-------|-------------|-----------|----------|
| M1 | BERT-base | FP32 | Pre-trained teacher (loaded, not trained here) |
| M2 | DistilBERT | FP32 | Supervised baseline (cross-entropy) |
| M3 | DistilBERT | FP32 | Knowledge distillation from M1 |
| M4 | DistilBERT | INT8 | Dynamic quantization of M2 |
| M5 | DistilBERT | INT8 | Dynamic quantization of M3 |

**Runtime → Change runtime type → T4 GPU before running.**

## Step 1 — Install dependencies

In [ ]:
import subprocess, sys

pkgs = [
    'transformers>=4.38.0',
    'datasets>=2.18.0',
    'accelerate>=0.27.0',
    'scikit-learn>=1.3.0',
    'matplotlib>=3.7.0',
    'pandas>=2.0.0',
    'numpy>=1.24.0',
]
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs,
                   capture_output=True, text=True)
if r.returncode != 0:
    print(r.stderr[-3000:])
    raise RuntimeError('pip install failed')
print('Dependencies ready.')

## Step 2 — Imports, seed, device

In [ ]:
import os, json, time, copy, random, warnings
from pathlib import Path
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')          # headless backend for Colab
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torch.cuda.amp import autocast, GradScaler

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    get_linear_schedule_with_warmup,
)
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score

warnings.filterwarnings('ignore', category=UserWarning)

# ── Reproducibility ────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# ── Device ─────────────────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = torch.cuda.is_available()   # mixed precision only on GPU

print(f'Device  : {DEVICE}')
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU     : {props.name}')
    print(f'VRAM    : {props.total_memory / 1024**3:.1f} GB')
print(f'AMP     : {USE_AMP}')
print(f'PyTorch : {torch.__version__}')

## Step 3 — Configuration

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────
BASE_DIR    = Path('/content/kdq_demo')   # all outputs go here
MODELS_DIR  = BASE_DIR / 'models'
RESULTS_DIR = BASE_DIR / 'results'
for d in [MODELS_DIR, RESULTS_DIR,
          MODELS_DIR / 'm2_baseline',
          MODELS_DIR / 'm3_distilled']:
    d.mkdir(parents=True, exist_ok=True)

# ── Model IDs ──────────────────────────────────────────────────────────────
# Teacher: verified BERT-base fine-tuned on SST-2 (92.7% dev accuracy)
# Source: https://huggingface.co/textattack/bert-base-uncased-SST-2
TEACHER_MODEL_ID = 'textattack/bert-base-uncased-SST-2'

# Student base: pretrained DistilBERT (no SST-2 fine-tuning — both M2 & M3
# start from identical weights so the comparison is fair)
STUDENT_MODEL_ID = 'distilbert-base-uncased'
TOKENIZER_ID     = 'bert-base-uncased'    # BERT tokenizer for all models

# ── Dataset ────────────────────────────────────────────────────────────────
NUM_TRAIN  = 1500    # stratified subsample of SST-2 train (adjust 1000–2000)
NUM_VAL    = 400     # stratified subsample of SST-2 validation
MAX_LEN    = 128
NUM_LABELS = 2

# ── Training ───────────────────────────────────────────────────────────────
BATCH_SIZE    = 16
EPOCHS        = 3
LR            = 2e-5
WEIGHT_DECAY  = 0.01
WARMUP_RATIO  = 0.1

# ── Distillation ───────────────────────────────────────────────────────────
TEMPERATURE = 2.0
ALPHA       = 0.5    # 0.5 * KL_div + 0.5 * CE

print('Configuration set.')
print(f'  TEACHER  : {TEACHER_MODEL_ID}')
print(f'  STUDENT  : {STUDENT_MODEL_ID}')
print(f'  NUM_TRAIN: {NUM_TRAIN}  NUM_VAL: {NUM_VAL}')
print(f'  EPOCHS   : {EPOCHS}  BATCH_SIZE: {BATCH_SIZE}')
print(f'  TEMP     : {TEMPERATURE}  ALPHA: {ALPHA}')
print(f'  Outputs  : {BASE_DIR}')

## Step 4 — Load SST-2 and prepare DataLoaders

In [ ]:
# ── Load raw dataset ───────────────────────────────────────────────────────
print('Loading GLUE SST-2…')
raw = load_dataset('glue', 'sst2')
print(f'  Raw train : {len(raw["train"]):,}')
print(f'  Raw val   : {len(raw["validation"]):,}')

# ── Stratified subsample ───────────────────────────────────────────────────
def stratified_sample(hf_split, n: int, seed: int = 42) -> pd.DataFrame:
    """Return n rows stratified by label."""
    df = hf_split.to_pandas()
    n_per = max(1, n // 2)
    return (
        df.groupby('label', group_keys=False)
          .apply(lambda g: g.sample(min(len(g), n_per), random_state=seed),
                 include_groups=False)
          .assign(label=lambda x: df.loc[x.index, 'label'].values)
          .reset_index(drop=True)
    )

train_df = stratified_sample(raw['train'],      NUM_TRAIN, SEED)
val_df   = stratified_sample(raw['validation'], NUM_VAL,   SEED)

print(f'\nSubsampled train : {len(train_df)}')
print(f'  labels: {train_df["label"].value_counts().sort_index().to_dict()}')
print(f'Subsampled val   : {len(val_df)}')
print(f'  labels: {val_df["label"].value_counts().sort_index().to_dict()}')

In [ ]:
# ── Tokenizer ──────────────────────────────────────────────────────────────
print(f'Loading tokenizer: {TOKENIZER_ID}…')
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_ID)
print(f'  vocab_size={tokenizer.vocab_size:,}')

# ── PyTorch Dataset ────────────────────────────────────────────────────────
class SST2Dataset(Dataset):
    def __init__(self, df: pd.DataFrame, tokenizer, max_len: int):
        self.sentences = df['sentence'].tolist()
        self.labels    = df['label'].tolist()
        self.tok       = tokenizer
        self.max_len   = max_len

    def __len__(self):  return len(self.sentences)

    def __getitem__(self, idx):
        enc = self.tok(
            self.sentences[idx],
            truncation=True,
            max_length=self.max_len,
            padding=False,
            return_tensors=None,
        )
        return {'input_ids': enc['input_ids'],
                'attention_mask': enc['attention_mask'],
                'labels': self.labels[idx]}

collator = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors='pt')

train_ds = SST2Dataset(train_df, tokenizer, MAX_LEN)
val_ds   = SST2Dataset(val_df,   tokenizer, MAX_LEN)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=collator, pin_memory=USE_AMP)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False,
                          collate_fn=collator, pin_memory=USE_AMP)

# Quick sanity check
b = next(iter(train_loader))
print(f'\nBatch check — input_ids: {tuple(b["input_ids"].shape)},  '
      f'labels: {b["labels"][:6].tolist()}')

## Step 5 — Load M1 teacher (pre-trained, no training needed)

In [ ]:
print(f'Loading teacher: {TEACHER_MODEL_ID}')
print('(This is a verified BERT-base checkpoint fine-tuned on SST-2.)')

try:
    teacher = AutoModelForSequenceClassification.from_pretrained(
        TEACHER_MODEL_ID,
        num_labels=NUM_LABELS,
    )
except Exception as e:
    raise RuntimeError(
        f'Could not load teacher {TEACHER_MODEL_ID}.\n'
        f'Error: {e}\n'
        'Alternatives:\n'
        '  1. Use "gchhablani/bert-large-cased-cased-finetuned-sst2" (larger)\n'
        '  2. Use "yoshitomo-matsubara/bert-base-uncased-sst2" \n'
        '  3. Set TEACHER_MODEL_ID to any HF model fine-tuned on SST-2.'
    ) from e

teacher.eval()
teacher.to(DEVICE)

# Count parameters
n_params = sum(p.numel() for p in teacher.parameters())
size_mb  = sum(p.numel() * p.element_size() for p in teacher.parameters()) / 1024**2
print(f'  Parameters : {n_params:,}')
print(f'  Size (FP32): {size_mb:.1f} MB')

# Quick inference test
with torch.no_grad():
    test_batch = next(iter(val_loader))
    test_batch = {k: v.to(DEVICE) for k, v in test_batch.items() if k != 'labels'}
    logits = teacher(**test_batch).logits
    preds  = logits.argmax(-1).cpu().tolist()
print(f'  Inference test — first 8 predictions: {preds[:8]}')
print('Teacher loaded and verified.')

## Step 6 — Shared training utilities

In [ ]:
# ── Evaluation ─────────────────────────────────────────────────────────────
@torch.no_grad()
def evaluate(model, loader, device=DEVICE) -> Dict:
    model.eval()
    all_preds, all_labels = [], []
    for batch in loader:
        labels = batch.pop('labels').to(device)
        batch  = {k: v.to(device) for k, v in batch.items()}
        logits = model(**batch).logits
        all_preds.extend(logits.argmax(-1).cpu().tolist())
        all_labels.extend(labels.cpu().tolist())
    acc = accuracy_score(all_labels, all_preds)
    f1  = f1_score(all_labels, all_preds, average='macro')
    return {'accuracy': round(acc, 4), 'f1': round(f1, 4),
            'n': len(all_labels)}

# ── Model size ─────────────────────────────────────────────────────────────
def model_size_mb(model) -> float:
    """FP32/INT8 parameter footprint in MB."""
    total = 0
    for p in model.parameters():
        total += p.numel() * p.element_size()
    # Also count buffers (quantised models store INT8 weights as buffers)
    for b in model.buffers():
        total += b.numel() * b.element_size()
    return round(total / 1024**2, 2)

# ── Latency benchmark ──────────────────────────────────────────────────────
def measure_latency_ms(model, loader, n_batches: int = 20,
                        device=torch.device('cpu')) -> float:
    """Mean per-batch CPU inference latency in ms over n_batches."""
    model = model.to(device)
    model.eval()
    times = []
    with torch.no_grad():
        for i, batch in enumerate(loader):
            if i >= n_batches:
                break
            batch = {k: v.to(device) for k, v in batch.items() if k != 'labels'}
            t0 = time.perf_counter()
            _ = model(**batch).logits
            times.append((time.perf_counter() - t0) * 1000)
    return round(float(np.mean(times)), 2)

# ── Build optimizer + scheduler ────────────────────────────────────────────
def build_optimizer_scheduler(model, n_steps: int):
    no_decay = ['bias', 'LayerNorm.weight']
    params = [
        {'params': [p for n, p in model.named_parameters()
                    if not any(nd in n for nd in no_decay)],
         'weight_decay': WEIGHT_DECAY},
        {'params': [p for n, p in model.named_parameters()
                    if any(nd in n for nd in no_decay)],
         'weight_decay': 0.0},
    ]
    opt = torch.optim.AdamW(params, lr=LR)
    warmup = int(WARMUP_RATIO * n_steps)
    sched  = get_linear_schedule_with_warmup(opt, warmup, n_steps)
    return opt, sched

print('Utilities defined.')

## Step 7 — Train M2: Supervised DistilBERT baseline (FP32)

In [ ]:
print('=' * 55)
print('  Training M2 — Supervised DistilBERT baseline')
print('=' * 55)

# Load fresh DistilBERT (pretrained, not SST-2 fine-tuned)
m2 = AutoModelForSequenceClassification.from_pretrained(
    STUDENT_MODEL_ID, num_labels=NUM_LABELS
)
m2.to(DEVICE)

# Save initial weights — M3 will start from an identical copy
m2_init_state = copy.deepcopy(m2.state_dict())
print(f'M2 loaded: {sum(p.numel() for p in m2.parameters()):,} parameters')

n_steps   = EPOCHS * len(train_loader)
opt2, sch2 = build_optimizer_scheduler(m2, n_steps)
scaler2   = GradScaler(enabled=USE_AMP)
ce_loss   = nn.CrossEntropyLoss()

m2_history = []
t_start = time.time()

for epoch in range(1, EPOCHS + 1):
    m2.train()
    epoch_loss, n_correct, n_total = 0.0, 0, 0

    for step, batch in enumerate(train_loader, 1):
        labels = batch.pop('labels').to(DEVICE)
        batch  = {k: v.to(DEVICE) for k, v in batch.items()}

        opt2.zero_grad()
        with autocast(enabled=USE_AMP):
            logits = m2(**batch).logits
            loss   = ce_loss(logits, labels)

        scaler2.scale(loss).backward()
        scaler2.unscale_(opt2)
        nn.utils.clip_grad_norm_(m2.parameters(), 1.0)
        scaler2.step(opt2)
        scaler2.update()
        sch2.step()

        epoch_loss += loss.item()
        n_correct  += (logits.argmax(-1) == labels).sum().item()
        n_total    += labels.size(0)

        if step % 20 == 0 or step == len(train_loader):
            print(f'  Epoch {epoch}/{EPOCHS}  step {step}/{len(train_loader)}'
                  f'  loss={epoch_loss/step:.4f}'
                  f'  train_acc={n_correct/n_total:.3f}')

    val_metrics = evaluate(m2, val_loader)
    elapsed = time.time() - t_start
    m2_history.append({'epoch': epoch, 'train_loss': epoch_loss / len(train_loader),
                        'train_acc': n_correct / n_total, **val_metrics})
    print(f'  → Epoch {epoch} done  val_acc={val_metrics["accuracy"]:.4f}'
          f'  val_f1={val_metrics["f1"]:.4f}  elapsed={elapsed:.0f}s')

# Save checkpoint
m2.save_pretrained(MODELS_DIR / 'm2_baseline')
tokenizer.save_pretrained(MODELS_DIR / 'm2_baseline')
print(f'\nM2 checkpoint saved to {MODELS_DIR}/m2_baseline')
print(f'Final M2 val_acc={m2_history[-1]["accuracy"]}')

## Step 8 — Train M3: Knowledge-distilled DistilBERT (FP32)

In [ ]:
print('=' * 55)
print('  Training M3 — Knowledge Distillation from M1')
print('=' * 55)
print(f'  alpha={ALPHA}  temperature={TEMPERATURE}')

# M3 starts from the SAME initial weights as M2 (fair comparison)
m3 = AutoModelForSequenceClassification.from_pretrained(
    STUDENT_MODEL_ID, num_labels=NUM_LABELS
)
m3.load_state_dict(m2_init_state)
m3.to(DEVICE)

# Teacher stays frozen
teacher.eval()
for p in teacher.parameters():
    p.requires_grad = False

n_steps   = EPOCHS * len(train_loader)
opt3, sch3 = build_optimizer_scheduler(m3, n_steps)
scaler3   = GradScaler(enabled=USE_AMP)
ce_loss   = nn.CrossEntropyLoss()

def distillation_loss(student_logits, teacher_logits, labels, T, alpha):
    """
    loss = alpha * T^2 * KL(softmax(student/T) || softmax(teacher/T))
         + (1 - alpha) * CrossEntropy(student, hard_labels)
    """
    # Soft targets
    p_teacher = F.softmax(teacher_logits / T, dim=-1)
    log_p_stu = F.log_softmax(student_logits / T, dim=-1)
    kl = F.kl_div(log_p_stu, p_teacher, reduction='batchmean') * (T ** 2)
    # Hard targets
    ce = ce_loss(student_logits, labels)
    return alpha * kl + (1.0 - alpha) * ce

m3_history = []
t_start = time.time()

for epoch in range(1, EPOCHS + 1):
    m3.train()
    epoch_loss, n_correct, n_total = 0.0, 0, 0

    for step, batch in enumerate(train_loader, 1):
        labels = batch.pop('labels').to(DEVICE)
        batch  = {k: v.to(DEVICE) for k, v in batch.items()}

        opt3.zero_grad()
        with autocast(enabled=USE_AMP):
            student_logits = m3(**batch).logits
            with torch.no_grad():
                teacher_logits = teacher(**batch).logits
            loss = distillation_loss(
                student_logits, teacher_logits, labels, TEMPERATURE, ALPHA
            )

        scaler3.scale(loss).backward()
        scaler3.unscale_(opt3)
        nn.utils.clip_grad_norm_(m3.parameters(), 1.0)
        scaler3.step(opt3)
        scaler3.update()
        sch3.step()

        epoch_loss += loss.item()
        n_correct  += (student_logits.argmax(-1) == labels).sum().item()
        n_total    += labels.size(0)

        if step % 20 == 0 or step == len(train_loader):
            print(f'  Epoch {epoch}/{EPOCHS}  step {step}/{len(train_loader)}'
                  f'  loss={epoch_loss/step:.4f}'
                  f'  train_acc={n_correct/n_total:.3f}')

    val_metrics = evaluate(m3, val_loader)
    elapsed = time.time() - t_start
    m3_history.append({'epoch': epoch, 'train_loss': epoch_loss / len(train_loader),
                        'train_acc': n_correct / n_total, **val_metrics})
    print(f'  → Epoch {epoch} done  val_acc={val_metrics["accuracy"]:.4f}'
          f'  val_f1={val_metrics["f1"]:.4f}  elapsed={elapsed:.0f}s')

# Save checkpoint
m3.save_pretrained(MODELS_DIR / 'm3_distilled')
tokenizer.save_pretrained(MODELS_DIR / 'm3_distilled')
print(f'\nM3 checkpoint saved to {MODELS_DIR}/m3_distilled')
print(f'Final M3 val_acc={m3_history[-1]["accuracy"]}')

## Step 9 — Apply INT8 dynamic quantization (M4, M5)

In [ ]:
print('Applying dynamic INT8 quantization…')

def quantize_model(fp32_model) -> nn.Module:
    """Dynamic INT8 quantization targeting Linear layers (CPU only)."""
    model_cpu = copy.deepcopy(fp32_model).cpu()
    q = torch.quantization.quantize_dynamic(
        model_cpu,
        {nn.Linear},
        dtype=torch.qint8,
    )
    return q

m2_cpu = copy.deepcopy(m2).cpu()
m3_cpu = copy.deepcopy(m3).cpu()

print('  Quantizing M2 → M4…')
m4 = quantize_model(m2_cpu)
print('  Quantizing M3 → M5…')
m5 = quantize_model(m3_cpu)

# Save quantized models as TorchScript state dicts
torch.save(m4, MODELS_DIR / 'm4_quantized_baseline.pt')
torch.save(m5, MODELS_DIR / 'm5_quantized_distilled.pt')

print(f'  M4 saved → {MODELS_DIR}/m4_quantized_baseline.pt')
print(f'  M5 saved → {MODELS_DIR}/m5_quantized_distilled.pt')
print('Quantization complete.')

## Step 10 — Evaluate all five models

In [ ]:
print('=' * 55)
print('  Evaluating all five models on the same val set')
print('=' * 55)

# Latency is always measured on CPU (quantization is CPU-only)
CPU = torch.device('cpu')

models_to_eval = [
    ('M1', 'BERT-base teacher',         'FP32', teacher.cpu()),
    ('M2', 'DistilBERT baseline',        'FP32', m2.cpu()),
    ('M3', 'DistilBERT distilled',       'FP32', m3.cpu()),
    ('M4', 'DistilBERT quant-baseline',  'INT8', m4),
    ('M5', 'DistilBERT quant-distilled', 'INT8', m5),
]

results = []
for mid, label, precision, model in models_to_eval:
    print(f'\n  Evaluating {mid} ({label})…')
    metrics = evaluate(model, val_loader, device=CPU)
    size    = model_size_mb(model)
    latency = measure_latency_ms(model, val_loader, n_batches=20, device=CPU)

    row = {
        'model':     mid,
        'label':     label,
        'precision': precision,
        'accuracy':  metrics['accuracy'],
        'f1_macro':  metrics['f1'],
        'size_mb':   size,
        'latency_ms': latency,
        'n_val':     metrics['n'],
    }
    results.append(row)
    print(f'    acc={metrics["accuracy"]:.4f}  f1={metrics["f1"]:.4f}  '
          f'size={size:.1f}MB  latency={latency:.1f}ms/batch')

# Move GPU models back for checkpoint saving
teacher.to(DEVICE)
m2.to(DEVICE)
m3.to(DEVICE)

print('\nAll evaluations complete.')

## Step 11 — Save results CSV and display table

In [ ]:
results_df = pd.DataFrame(results)

# Add derived columns for the presentation
m1_acc = results_df.loc[results_df['model'] == 'M1', 'accuracy'].iloc[0]
results_df['accuracy_vs_m1'] = (results_df['accuracy'] - m1_acc).round(4)

m2_size = results_df.loc[results_df['model'] == 'M2', 'size_mb'].iloc[0]
results_df['compression_vs_m2'] = (m2_size / results_df['size_mb']).round(2)

# Save CSV
csv_path = RESULTS_DIR / 'model_comparison.csv'
results_df.to_csv(csv_path, index=False)
print(f'Results saved to {csv_path}\n')
print(results_df[['model', 'label', 'precision', 'accuracy', 'f1_macro',
                   'size_mb', 'latency_ms']].to_string(index=False))

## Step 12 — Comparison plots

In [ ]:
COLORS = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0', '#F44336']
LABELS = results_df['model'].tolist()

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle(
    'Knowledge Distillation + Quantization — Model Comparison',
    fontsize=14, fontweight='bold'
)

# ── Accuracy ───────────────────────────────────────────────────────────────
ax = axes[0]
bars = ax.bar(LABELS, results_df['accuracy'], color=COLORS, width=0.6, edgecolor='black')
ax.set_title('Validation Accuracy')
ax.set_ylabel('Accuracy')
ax.set_ylim(max(0, results_df['accuracy'].min() - 0.05), 1.0)
ax.axhline(m1_acc, color='gray', linestyle='--', linewidth=1, label='Teacher')
for bar, val in zip(bars, results_df['accuracy']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{val:.3f}', ha='center', va='bottom', fontsize=9)
ax.legend(fontsize=8)

# ── Model size ─────────────────────────────────────────────────────────────
ax = axes[1]
bars = ax.bar(LABELS, results_df['size_mb'], color=COLORS, width=0.6, edgecolor='black')
ax.set_title('Model Size (MB)')
ax.set_ylabel('Size (MB)')
for bar, val in zip(bars, results_df['size_mb']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{val:.0f}', ha='center', va='bottom', fontsize=9)

# ── CPU Latency ────────────────────────────────────────────────────────────
ax = axes[2]
bars = ax.bar(LABELS, results_df['latency_ms'], color=COLORS, width=0.6, edgecolor='black')
ax.set_title(f'CPU Inference Latency (ms/batch={32})')
ax.set_ylabel('Latency (ms)')
for bar, val in zip(bars, results_df['latency_ms']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val:.0f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plot_path = RESULTS_DIR / 'model_comparison.png'
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Plot saved to {plot_path}')

In [ ]:
# Training curves for M2 and M3
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Training Curves — M2 vs M3', fontsize=13, fontweight='bold')

epochs_x = list(range(1, EPOCHS + 1))

ax = axes[0]
ax.plot(epochs_x, [h['train_loss'] for h in m2_history], 'b-o', label='M2 baseline')
ax.plot(epochs_x, [h['train_loss'] for h in m3_history], 'r-o', label='M3 distilled')
ax.set_title('Training Loss per Epoch')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(epochs_x, [h['accuracy'] for h in m2_history], 'b-o', label='M2 baseline')
ax.plot(epochs_x, [h['accuracy'] for h in m3_history], 'r-o', label='M3 distilled')
ax.axhline(m1_acc, color='gray', linestyle='--', linewidth=1, label='M1 teacher')
ax.set_title('Validation Accuracy per Epoch')
ax.set_xlabel('Epoch')
ax.set_ylabel('Accuracy')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
curves_path = RESULTS_DIR / 'training_curves.png'
plt.savefig(curves_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Training curves saved to {curves_path}')

## Step 13 — Save all results and package for Streamlit

In [ ]:
# ── Summary JSON (Streamlit demo reads this) ───────────────────────────────
summary = {
    'experiment': 'Knowledge Distillation + Quantization for Compact Models',
    'dataset': 'GLUE SST-2',
    'n_train': NUM_TRAIN,
    'n_val':   NUM_VAL,
    'epochs':  EPOCHS,
    'alpha':   ALPHA,
    'temperature': TEMPERATURE,
    'seed': SEED,
    'models': results,
    'm2_training_history': m2_history,
    'm3_training_history': m3_history,
}
json_path = RESULTS_DIR / 'experiment_summary.json'
with open(json_path, 'w') as f:
    json.dump(summary, f, indent=2)
print(f'Summary JSON saved to {json_path}')

# ── List all saved artefacts ───────────────────────────────────────────────
print('\nAll artefacts:')
for p in sorted(BASE_DIR.rglob('*')):
    if p.is_file() and not p.name.startswith('.'):
        size_kb = p.stat().st_size / 1024
        print(f'  {str(p.relative_to(BASE_DIR)):<50} {size_kb:>8.1f} KB')

In [ ]:
import shutil

# Zip results for download
zip_path = '/content/kdq_results.zip'
shutil.make_archive('/content/kdq_results', 'zip',
                    root_dir=str(BASE_DIR), base_dir='.')
size_mb = Path(zip_path).stat().st_size / 1024**2
print(f'Results zip: {zip_path}  ({size_mb:.1f} MB)')

# Trigger download
try:
    from google.colab import files as _colab_files
    _colab_files.download(zip_path)
    print('Download started — save kdq_results.zip to your local machine.')
except ImportError:
    print(f'Not in Colab — results are at {zip_path}')

## Step 14 — Final summary

In [ ]:
print('\n' + '=' * 65)
print('  EXPERIMENT COMPLETE — RESULTS SUMMARY')
print('=' * 65)

col_w = [5, 28, 6, 10, 10, 10, 13]
header = ['Model', 'Description', 'Prec.', 'Acc.', 'F1', 'Size MB', 'Latency ms']
row_fmt = '  '.join(f'{{:<{w}}}' for w in col_w)
print(row_fmt.format(*header))
print('  ' + '-' * 60)
for r in results:
    print(row_fmt.format(
        r['model'], r['label'], r['precision'],
        str(r['accuracy']), str(r['f1_macro']),
        str(r['size_mb']), str(r['latency_ms'])
    ))

print('=' * 65)
best_acc = max(results, key=lambda r: r['accuracy'])
best_lat = min(results, key=lambda r: r['latency_ms'])
best_siz = min(results, key=lambda r: r['size_mb'])
print(f'  Highest accuracy : {best_acc["model"]} ({best_acc["accuracy"]})')
print(f'  Lowest latency   : {best_lat["model"]} ({best_lat["latency_ms"]} ms)')
print(f'  Smallest size    : {best_siz["model"]} ({best_siz["size_mb"]} MB)')
print('=' * 65)
print('\nDownload kdq_results.zip and copy to your local project:')
print('  results/model_comparison.csv')
print('  results/model_comparison.png')
print('  results/training_curves.png')
print('  results/experiment_summary.json')
print('  models/m2_baseline/')
print('  models/m3_distilled/')
print('  models/m4_quantized_baseline.pt')
print('  models/m5_quantized_distilled.pt')